# Task 8 - Fine-tune T5 (text-to-text)

Thay vì gắn classification head, cách này coi mọi subtask là bài toán sinh văn bản: đưa một prompt mô tả nhiệm vụ, model sinh ra trực tiếp nhãn dạng text. Cùng một kiến trúc T5 dùng được cho cả phân loại nhị phân, đa lớp lẫn đa nhãn — chỉ cần đổi prompt và cách parse output, không cần đổi kiến trúc model.

In [ ]:
!pip install -q transformers datasets accelerate sentencepiece scikit-learn

In [ ]:
!git clone -q https://github.com/hoanhh4977/SemEval2027.git
%cd SemEval2027/

In [ ]:
import pandas as pd

data = pd.read_csv("data/task8/train/StereoQueerEval_EN_training.tsv", sep="\t")
data.head()

## Chuẩn bị prompt (input) và nhãn dạng text (target) cho 3 subtask

Khác với các notebook phân loại (nhãn là số 0/1/2), ở đây input là một câu prompt mô tả nhiệm vụ nối với comment, còn target là chuỗi text model phải sinh ra.

In [ ]:
# Subtask A - Stereotype Detection
PREFIX_A = "stereotype detection: "
data["input_A"] = PREFIX_A + data["yt_comment"]
data["target_A"] = data["stereotype"]

# Subtask B - Hate Speech Classification
PREFIX_B = "hate speech classification: "
hate_label_map = {"no": "no", "yes_implicit": "implicit", "yes_explicit": "explicit"}
data["input_B"] = PREFIX_B + data["yt_comment"]
data["target_B"] = data["hate_speech"].map(hate_label_map)

# Subtask C - Target & Identity Identification (multilabel, chỉ trên comment hateful)
PREFIX_C = "target identification: "

def parse_target(x):
    if x == "none":
        return []
    scope, identities = x.split("_", 1)
    return [scope] + identities.split(",")

data["target_labels"] = data["target"].apply(parse_target)
data_c = data[data["hate_speech"] != "no"].reset_index(drop=True)
data_c["input_C"] = PREFIX_C + data_c["yt_comment"]
data_c["target_C"] = data_c["target_labels"].apply(lambda labels: ", ".join(labels))
print(f"Subtask C chỉ dùng {len(data_c)}/{len(data)} comment hateful")

## Chia Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

train_A, test_A = train_test_split(data, test_size=0.2, random_state=42, stratify=data["stereotype"])
train_B, test_B = train_test_split(data, test_size=0.2, random_state=42, stratify=data["hate_speech"])
train_C, test_C = train_test_split(data_c, test_size=0.2, random_state=42)

print(f"A: {len(train_A)}/{len(test_A)}  B: {len(train_B)}/{len(test_B)}  C: {len(train_C)}/{len(test_C)}")

## Hàm huấn luyện dùng chung (Seq2Seq)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from datasets import Dataset, DatasetDict

gen_tokenizer = AutoTokenizer.from_pretrained("t5-base")

def preprocess_seq2seq(batch):
    model_inputs = gen_tokenizer(batch["input_text"], truncation=True, max_length=128)
    labels = gen_tokenizer(text_target=batch["target_text"], truncation=True, max_length=16)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

def build_seq2seq_dataset(train_df, test_df, input_col, target_col):
    dd = DatasetDict({
        "train": Dataset.from_pandas(
            train_df[[input_col, target_col]].rename(
                columns={input_col: "input_text", target_col: "target_text"}),
            preserve_index=False,
        ),
        "test": Dataset.from_pandas(
            test_df[[input_col, target_col]].rename(
                columns={input_col: "input_text", target_col: "target_text"}),
            preserve_index=False,
        ),
    })
    return dd.map(preprocess_seq2seq, remove_columns=["input_text", "target_text"])

def train_t5(train_df, test_df, input_col, target_col, run_name):
    dataset_dict = build_seq2seq_dataset(train_df, test_df, input_col, target_col)
    model = AutoModelForSeq2SeqLM.from_pretrained("t5-base")
    collator = DataCollatorForSeq2Seq(tokenizer=gen_tokenizer, model=model)
    args = Seq2SeqTrainingArguments(
        output_dir=run_name, learning_rate=3e-4,
        per_device_train_batch_size=8, per_device_eval_batch_size=8,
        num_train_epochs=3, weight_decay=0.01,
        predict_with_generate=True, generation_max_length=16,
        eval_strategy="epoch", save_strategy="no", report_to="none",
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=dataset_dict["train"], eval_dataset=dataset_dict["test"],
        processing_class=gen_tokenizer, data_collator=collator,
    )
    trainer.train()
    return trainer, dataset_dict

## Hàm đánh giá dùng chung

Giải mã (decode) chuỗi model sinh ra rồi so với nhãn gốc. Với subtask multilabel, tách chuỗi theo dấu phẩy thành tập nhãn rồi tính F1 như các notebook trước.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

def evaluate_generation(trainer, dataset_dict, multilabel=False, all_labels=None):
    preds_output = trainer.predict(dataset_dict["test"])

    pred_ids = np.where(preds_output.predictions != -100, preds_output.predictions, gen_tokenizer.pad_token_id)
    decoded_preds = [t.strip().lower() for t in gen_tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]

    label_ids = np.where(preds_output.label_ids != -100, preds_output.label_ids, gen_tokenizer.pad_token_id)
    decoded_labels = [t.strip().lower() for t in gen_tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

    if multilabel:
        mlb = MultiLabelBinarizer(classes=all_labels)
        pred_sets = [[x.strip() for x in p.split(",") if x.strip()] for p in decoded_preds]
        gold_sets = [[x.strip() for x in g.split(",") if x.strip()] for g in decoded_labels]
        y_pred = mlb.fit_transform(pred_sets)
        y_true = mlb.transform(gold_sets)
        print(f"F1-micro: {f1_score(y_true, y_pred, average='micro'):.3f}  "
              f"F1-macro: {f1_score(y_true, y_pred, average='macro'):.3f}")
    else:
        acc = accuracy_score(decoded_labels, decoded_preds)
        print(f"Accuracy (so khớp chuỗi text): {acc:.3f}")

    return decoded_preds, decoded_labels

## Subtask A: Stereotype Detection

In [ ]:
trainer_A, ds_A = train_t5(train_A, test_A, "input_A", "target_A", "t5_subtaskA")
evaluate_generation(trainer_A, ds_A)

## Subtask B: Hate Speech Classification

In [ ]:
trainer_B, ds_B = train_t5(train_B, test_B, "input_B", "target_B", "t5_subtaskB")
evaluate_generation(trainer_B, ds_B)

## Subtask C: Target & Identity Identification (multilabel)

In [ ]:
all_labels_C = sorted(set(l for labels in data_c["target_labels"] for l in labels))
trainer_C, ds_C = train_t5(train_C, test_C, "input_C", "target_C", "t5_subtaskC")
evaluate_generation(trainer_C, ds_C, multilabel=True, all_labels=all_labels_C)

Lưu ý: dùng `t5-base` nguyên bản (chưa chỉ lệnh/instruction-tuned) nên bắt buộc phải fine-tune mới dùng được tốt, không nên chỉ prompt zero-shot như FLAN-T5. Huấn luyện 3 model lần lượt khá tốn thời gian trên GPU free của Colab — nếu muốn thử nhanh trước, có thể đổi `"t5-base"` thành `"t5-small"` ở hàm `train_t5`, rồi đổi lại khi đã chạy ổn.